In [1]:
!pip install pandas requests sqlalchemy mysql-connector-python python-dotenv

# Task 3: Build a Modular, Logged ETL System

This notebook implements a modular, logged ETL (Extract, Transform, Load) system based on the requirements provided. The system is designed with four independent functions: `extract()`, `clean()`, `transform()`, and `load()`. It features error handling, data enrichment, summary generation, and idempotent loading to both CSV and SQLite.

In [2]:
import pandas as pd
import requests
import logging
import os

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [3]:
from dotenv import load_dotenv
import mysql.connector

# Load environment variables from .env file
load_dotenv()

# Define constants for MySQL connection
DB_HOST = os.getenv('DB_HOST')
DB_USER = os.getenv('DB_USER')
DB_PORT = int(os.getenv('DB_PORT', 3306)) # Default to 3306 if not specified
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_NAME = 'etl_database' # Define a database name for MySQL

# Existing constants
CSV_FILE = 'posts_data.csv'
TABLE_NAME = 'posts'

In [4]:
pip install pandas requests sqlalchemy mysql-connector-python python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 1. `extract()` Function

This function is responsible for fetching raw data from a public API. It includes comprehensive error handling for network issues, HTTP status codes, and timeouts. It logs the number of records extracted.

In [5]:
def extract(url: str = 'https://jsonplaceholder.typicode.com/posts') -> pd.DataFrame:
    logging.info(f"Starting data extraction from {url}")
    try:
        response = requests.get(url, timeout=10) # 10-second timeout
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        data = response.json()
        df = pd.DataFrame(data)
        logging.info(f"Successfully extracted {len(df)} rows.")
        return df
    except requests.exceptions.Timeout:
        logging.error(f"Extraction failed: Request timed out after 10 seconds from {url}")
        return pd.DataFrame() # Return empty DataFrame on failure
    except requests.exceptions.RequestException as e:
        logging.error(f"Extraction failed due to network or HTTP error from {url}: {e}")
        return pd.DataFrame() # Return empty DataFrame on failure
    except ValueError as e:
        logging.error(f"Extraction failed: Could not parse JSON response from {url}: {e}")
        return pd.DataFrame() # Return empty DataFrame on failure


### 2. `clean()` Function

This function performs basic data cleaning, such as handling missing values. For the JSONPlaceholder API, there aren't typically missing values, but the structure is included for robustness. It logs the number of rows processed and outputted.

In [6]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    logging.info(f"Starting data cleaning. Received {len(df)} rows.")
    if df.empty:
        logging.warning("No data to clean. Returning empty DataFrame.")
        return pd.DataFrame()

    # Example: Drop rows with any missing values. Adjust strategy as needed.
    initial_rows = len(df)
    df_cleaned = df.dropna().copy() # Using .copy() to avoid SettingWithCopyWarning

    # Further cleaning steps can be added here, e.g., type conversions
    df_cleaned['userId'] = df_cleaned['userId'].astype(int)
    df_cleaned['id'] = df_cleaned['id'].astype(int)

    logging.info(f"Finished data cleaning. Outputted {len(df_cleaned)} rows. Dropped {initial_rows - len(df_cleaned)} rows due to missing values.")
    return df_cleaned


### 3. `transform()` Function

This function enriches the data by engineering new calculated columns and generates a summary table using `groupby()`. It logs the transformation steps and the number of rows processed.

In [7]:
def transform(df: pd.DataFrame) -> pd.DataFrame:
    logging.info(f"Starting data transformation. Received {len(df)} rows.")
    if df.empty:
        logging.warning("No data to transform. Returning empty DataFrame.")
        return pd.DataFrame()

    df_transformed = df.copy()

    # Engineer at least 3 new calculated columns
    # 1. title_word_count
    df_transformed['title_word_count'] = df_transformed['title'].apply(lambda x: len(str(x).split()))
    logging.info("Added 'title_word_count' column.")

    # 2. body_word_count
    df_transformed['body_word_count'] = df_transformed['body'].apply(lambda x: len(str(x).split()))
    logging.info("Added 'body_word_count' column.")

    # 3. title_body_ratio (handle division by zero)
    df_transformed['title_body_ratio'] = df_transformed.apply(
        lambda row: row['title_word_count'] / row['body_word_count'] if row['body_word_count'] > 0 else 0,
        axis=1
    )
    logging.info("Added 'title_body_ratio' column.")

    # Use groupby() to produce a printed summary table
    logging.info("Generating summary table by 'userId'.")
    summary_table = df_transformed.groupby('userId').agg(
        mean_title_word_count=('title_word_count', 'mean'),
        min_body_word_count=('body_word_count', 'min'),
        max_body_word_count=('body_word_count', 'max'),
        total_posts=('id', 'count')
    ).reset_index()

    print("\n--- Summary Table by User ID ---")
    print(summary_table.to_string()) # Use to_string() for full display in logs/output
    print("--------------------------------\n")
    logging.info("Generated and printed summary table.")

    logging.info(f"Finished data transformation. Outputted {len(df_transformed)} rows.")
    return df_transformed


### 4. `load()` Function

This function loads the processed data into two destinations: a CSV file and a SQLite database. It ensures idempotency for the SQLite database by using `if_exists='replace'` after removing old data, or `if_exists='append'` with a primary key constraint or `upsert` mechanism for more robust solutions (for this example, we'll ensure replacement for simplicity). It logs the loading process and the number of rows loaded.

In [8]:
# Old SQLite load function - no longer used.
# def load(df: pd.DataFrame, csv_file: str, sqlite_db: str, table_name: str):
#     logging.info(f"Starting data loading. Received {len(df)} rows.")
#     if df.empty:
#         logging.warning("No data to load. Skipping loading steps.")
#         return

#     # Load to CSV
#     try:
#         df.to_csv(csv_file, index=False)
#         logging.info(f"Successfully loaded {len(df)} rows to CSV: {csv_file}")
#     except Exception as e:
#         logging.error(f"Failed to load data to CSV {csv_file}: {e}")

#     # Load to SQLite with idempotency
#     try:
#         conn = sqlite3.connect(sqlite_db)
#         cursor = conn.cursor()

#         df.to_sql(name=table_name, con=conn, if_exists='replace', index=False, dtype={
#             'userId': 'INTEGER',
#             'id': 'INTEGER PRIMARY KEY',
#             'title': 'TEXT',
#             'body': 'TEXT',
#             'title_word_count': 'INTEGER',
#             'body_word_count': 'INTEGER',
#             'title_body_ratio': 'REAL'
#         })
#         logging.info(f"Successfully loaded {len(df)} rows to SQLite table '{table_name}' in {sqlite_db} (replaced existing data).")

#     except sqlite3.Error as e:
#         logging.error(f"Failed to load data to SQLite database {sqlite_db}: {e}")
#     finally:
#         if 'conn' in locals() and conn:
#             conn.close()
#     logging.info("Finished data loading.")

In [9]:
def load(df: pd.DataFrame, csv_file: str, db_host: str, db_user: str, db_password: str, db_name: str, table_name: str, db_port: int):
    logging.info(f"Starting data loading. Received {len(df)} rows.")
    if df.empty:
        logging.warning("No data to load. Skipping loading steps.")
        return

    # Load to CSV
    try:
        df.to_csv(csv_file, index=False)
        logging.info(f"Successfully loaded {len(df)} rows to CSV: {csv_file}")
    except Exception as e:
        logging.error(f"Failed to load data to CSV {csv_file}: {e}")

    # Load to MySQL with idempotency
    mysql_conn = None
    try:
        # Connect to MySQL
        mysql_conn = mysql.connector.connect(
            host=db_host,
            user=db_user,
            password=db_password,
            port=db_port
        )
        cursor = mysql_conn.cursor()

        # Create database if it doesn't exist
        cursor.execute(f"CREATE DATABASE IF NOT EXISTS {db_name}")
        mysql_conn.database = db_name # Switch to the new database

        # Create table with appropriate schema and primary key
        # For idempotency, we will TRUNCATE the table before inserting
        create_table_query = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            userId INT,
            id INT PRIMARY KEY,
            title TEXT,
            body TEXT,
            title_word_count INT,
            body_word_count INT,
            title_body_ratio REAL
        )
        """
        cursor.execute(create_table_query)
        logging.info(f"Table '{table_name}' ensured in MySQL database '{db_name}'.")

        # Truncate table for idempotency (clear existing data)
        cursor.execute(f"TRUNCATE TABLE {table_name}")
        logging.info(f"Truncated table '{table_name}' for idempotent loading.")

        # Insert data into MySQL
        # Prepare the insert query with placeholders
        cols = ', '.join(df.columns)
        placeholders = ', '.join(['%s'] * len(df.columns))
        insert_query = f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})"

        # Convert DataFrame rows to a list of tuples for executemany
        data_to_insert = [tuple(row) for row in df.to_numpy()]

        cursor.executemany(insert_query, data_to_insert)
        mysql_conn.commit()

        logging.info(f"Successfully loaded {len(df)} rows to MySQL table '{table_name}' in '{db_name}'.")

    except mysql.connector.Error as e:
        logging.error(f"Failed to load data to MySQL database '{db_name}': {e}")
        if mysql_conn and mysql_conn.is_connected():
            mysql_conn.rollback()
    finally:
        if mysql_conn and mysql_conn.is_connected():
            cursor.close()
            mysql_conn.close()
    logging.info("Finished data loading.")

### Main Execution Flow

This section orchestrates the execution of the ETL pipeline by calling the `extract()`, `clean()`, `transform()`, and `load()` functions sequentially. It demonstrates how the modular components work together.

In [10]:
import logging
import pandas as pd
import mysql.connector
from IPython.display import display

def run_etl_pipeline():
    logging.info("--- ETL Pipeline Started ---")

    # 1. Extract
    raw_df = extract()
    if raw_df.empty:
        logging.error("ETL Pipeline aborted: Extraction failed or returned no data.")
        return

    # 2. Clean
    cleaned_df = clean(raw_df)
    if cleaned_df.empty:
        logging.error("ETL Pipeline aborted: Cleaning failed or resulted in no data.")
        return

    # 3. Transform
    transformed_df = transform(cleaned_df)
    if transformed_df.empty:
        logging.error("ETL Pipeline aborted: Transformation failed or resulted in no data.")
        return

    # 4. Load
    # Call the MySQL load function with correct parameters
    load(transformed_df, CSV_FILE, DB_HOST, DB_USER, DB_PASSWORD, DB_NAME, TABLE_NAME, DB_PORT)

    logging.info("--- ETL Pipeline Finished ---")

# Run the pipeline
run_etl_pipeline()

# Verify MySQL load by reading back data
logging.info(f"Verifying data in MySQL database '{DB_NAME}' table '{TABLE_NAME}'.")
mysql_conn_verify = None
try:
    mysql_conn_verify = mysql.connector.connect(
        host=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME, # Connect directly to the database
        port=DB_PORT
    )
    verified_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME} LIMIT 5", mysql_conn_verify)
    print("\n--- Data from MySQL (first 5 rows) ---")
    display(verified_df)
    print("----------------------------------------\n")
except mysql.connector.Error as e:
    logging.error(f"Failed to read from MySQL for verification: {e}")
finally:
    if mysql_conn_verify and mysql_conn_verify.is_connected():
        mysql_conn_verify.close()

2026-05-15 12:59:09,244 - INFO - --- ETL Pipeline Started ---
2026-05-15 12:59:09,245 - INFO - Starting data extraction from https://jsonplaceholder.typicode.com/posts


2026-05-15 12:59:09,466 - INFO - Successfully extracted 100 rows.
2026-05-15 12:59:09,467 - INFO - Starting data cleaning. Received 100 rows.
2026-05-15 12:59:09,470 - INFO - Finished data cleaning. Outputted 100 rows. Dropped 0 rows due to missing values.
2026-05-15 12:59:09,470 - INFO - Starting data transformation. Received 100 rows.
2026-05-15 12:59:09,472 - INFO - Added 'title_word_count' column.
2026-05-15 12:59:09,473 - INFO - Added 'body_word_count' column.
2026-05-15 12:59:09,475 - INFO - Added 'title_body_ratio' column.
2026-05-15 12:59:09,475 - INFO - Generating summary table by 'userId'.
2026-05-15 12:59:09,485 - INFO - Generated and printed summary table.
2026-05-15 12:59:09,486 - INFO - Finished data transformation. Outputted 100 rows.
2026-05-15 12:59:09,486 - INFO - Starting data loading. Received 100 rows.
2026-05-15 12:59:09,490 - INFO - Successfully loaded 100 rows to CSV: posts_data.csv
2026-05-15 12:59:09,539 - INFO - Table 'posts' ensured in MySQL database 'etl_da


--- Summary Table by User ID ---
   userId  mean_title_word_count  min_body_word_count  max_body_word_count  total_posts
0       1                    5.3                   21                   31           10
1       2                    5.9                   18                   29           10
2       3                    6.4                   21                   27           10
3       4                    5.5                   25                   32           10
4       5                    6.8                   16                   29           10
5       6                    6.7                   17                   32           10
6       7                    5.8                   19                   31           10
7       8                    6.9                   16                   30           10
8       9                    6.5                   16                   28           10
9      10                    5.5                   17                   32           1

/tmp/ipykernel_81556/2090627900.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  verified_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME} LIMIT 5", mysql_conn_verify)


,userId,id,title,body,title_word_count,body_word_count,title_body_ratio
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...,9,23,0.391304
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...,3,31,0.096774
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...,9,26,0.346154
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...,4,28,0.142857
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...,3,23,0.130435


----------------------------------------

